In [1]:
import geopandas as gpd

FIELD_SHP = r"C:\Master Physische Geographie\Canada\Masterarbeit\thesis_backup\species_feld_merged.shp"
PLOTS_SHP = r"D:\thesis\Wa_layout\Rando_points.shp"

field = gpd.read_file(FIELD_SHP).to_crs("EPSG:32611")
plots = gpd.read_file(PLOTS_SHP).to_crs("EPSG:32611")

plots_buf = plots.copy()
plots_buf["geometry"] = plots.geometry.buffer(3.0)
joined = gpd.sjoin(field, plots_buf, how="inner", predicate="within")

# per-class counts per plot
counts = joined.groupby(["id", "class"]).size().unstack(fill_value=0)
counts.columns.name = None
for col in ["deciduous_tree", "shrub"]:
    if col not in counts.columns:
        counts[col] = 0

print(counts[["deciduous_tree", "shrub"]].to_string())

    deciduous_tree  shrub
id                       
0               17      2
1               12      0
2                0     14
3                0      8
4                3     10
5                0     12
6                0     11
7                0      1
8                0      0
9                2      2
10               0      3
11               0     27
12               0      3
13               5      0
14              19      0
15               1     15
16              12      0
17               8      0
18               0      0
19               0     12
20              27      0
21               0      0
22              12      0
23              26      0
24               0     18
25              11      2
26              31     15
27               0      2
28               0      4
29              13      0
30               1      2
31              12      0
32               0     12
33              10      1
34               0     15
35              10      0
36          

In [1]:
import os
import re
import pandas as pd
import numpy as np
from scipy import stats

PRED_DIR_SUMMER = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\field_plot_evaluation_v2\eval_summer\labels"
PRED_DIR_FALL   = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\field_plot_evaluation_v2\eval_fall\labels"
CONF_THRESHOLD  = 0.25

# per-class field counts from spatial join
field_deciduous = {
    0:17, 1:12, 2:0,  3:0,  4:3,  5:0,  6:0,  7:0,  8:0,  9:2,
    10:0, 11:0, 12:0, 13:5, 14:19,15:1, 16:12,17:8, 18:0, 19:0,
    20:27,21:0, 22:12,23:26,24:0, 25:11,26:31,27:0, 28:0, 29:13,
    30:1, 31:12,32:0, 33:10,34:0, 35:10,36:11,37:13,38:26,39:5,
    40:0, 41:4, 42:0, 43:10,44:3, 45:2, 46:8, 47:9, 48:0, 49:0,
}

field_shrub = {
    0:2,  1:0,  2:14, 3:8,  4:10, 5:12, 6:11, 7:1,  8:0,  9:2,
    10:3, 11:27,12:3, 13:0, 14:0, 15:15,16:0, 17:0, 18:0, 19:12,
    20:0, 21:0, 22:0, 23:0, 24:18,25:2, 26:15,27:2, 28:4, 29:0,
    30:2, 31:0, 32:12,33:1, 34:15,35:0, 36:4, 37:0, 38:1, 39:0,
    40:7, 41:2, 42:9, 43:0, 44:2, 45:7, 46:0, 47:0, 48:4, 49:15,
}

field_total = {k: field_deciduous[k] + field_shrub[k] for k in range(50)}

def count_predictions_filtered(pred_dir, conf_thresh):
    counts = {}
    for fname in os.listdir(pred_dir):
        if not fname.endswith(".txt"):
            continue
        match = re.search(r"_plot(\d+)_", fname)
        if not match:
            continue
        plot_id = int(match.group(1))

        with open(os.path.join(pred_dir, fname)) as f:
            lines = [l.strip() for l in f if l.strip()]

        n_dec = 0
        n_shr = 0
        for line in lines:
            parts = line.split()
            if len(parts) < 6:
                continue
            if float(parts[5]) < conf_thresh:
                continue
            if int(parts[0]) == 0:
                n_dec += 1
            elif int(parts[0]) == 1:
                n_shr += 1

        counts[plot_id] = {"dec": n_dec, "shr": n_shr, "total": n_dec + n_shr}
    return counts

def correlate(field_vals, pred_vals, label):
    pairs = [(f, p) for f, p in zip(field_vals, pred_vals) if f > 0 or p > 0]
    f_arr = np.array([x[0] for x in pairs])
    p_arr = np.array([x[1] for x in pairs])

    if f_arr.std() == 0 or p_arr.std() == 0:
        print(f"  {label}: constant input — correlation undefined")
        return

    r, p   = stats.pearsonr(f_arr, p_arr)
    rho, p_rho = stats.spearmanr(f_arr, p_arr)
    print(f"  {label}:")
    print(f"    Pearson r={r:.3f} (p={p:.4f}) {'*' if p<0.05 else ''}")
    print(f"    Spearman rho={rho:.3f} (p={p_rho:.4f}) {'*' if p_rho<0.05 else ''}")

def compute_all(pred_dir, season):
    print(f"\n{'='*55}")
    print(f"{season.upper()} — Per-class correlation (conf={CONF_THRESHOLD})")
    print(f"{'='*55}")

    preds = count_predictions_filtered(pred_dir, CONF_THRESHOLD)

    rows = []
    for pid in range(50):
        pred = preds.get(pid, {"dec": 0, "shr": 0, "total": 0})
        rows.append({
            "plot_id":      pid,
            "field_dec":    field_deciduous[pid],
            "field_shr":    field_shrub[pid],
            "field_total":  field_total[pid],
            "pred_dec":     pred["dec"],
            "pred_shr":     pred["shr"],
            "pred_total":   pred["total"],
        })

    df = pd.DataFrame(rows)

    # only plots with at least 1 field plant of that class
    print(f"\nTotal class correlation (all plots with >0 field plants):")
    valid = df[df["field_total"] > 0]
    correlate(valid["field_total"], valid["pred_total"], "Total")

    print(f"\nDeciduous_tree correlation (plots with >0 field deciduous):")
    dec_valid = df[df["field_dec"] > 0]
    correlate(dec_valid["field_dec"], dec_valid["pred_dec"], "deciduous_tree")

    print(f"\nShrub correlation (plots with >0 field shrubs):")
    shr_valid = df[df["field_shr"] > 0]
    correlate(shr_valid["field_shr"], shr_valid["pred_shr"], "shrub")

    print(f"\nDetection counts summary:")
    print(f"  Total field plants:   {df['field_total'].sum()}")
    print(f"  Total model preds:    {df['pred_total'].sum()}")
    print(f"  Field deciduous:      {df['field_dec'].sum()}")
    print(f"  Pred deciduous:       {df['pred_dec'].sum()}")
    print(f"  Field shrub:          {df['field_shr'].sum()}")
    print(f"  Pred shrub:           {df['pred_shr'].sum()}")

    return df

summer_df = compute_all(PRED_DIR_SUMMER, "summer")
fall_df   = compute_all(PRED_DIR_FALL,   "fall")

# save
output = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\field_plot_evaluation_v2\plot_level_correlation_perclass.csv"
merged = summer_df.rename(columns={
    "pred_dec": "summer_pred_dec",
    "pred_shr": "summer_pred_shr",
    "pred_total": "summer_pred_total"
}).merge(
    fall_df[["plot_id","pred_dec","pred_shr","pred_total"]].rename(columns={
        "pred_dec": "fall_pred_dec",
        "pred_shr": "fall_pred_shr",
        "pred_total": "fall_pred_total"
    }), on="plot_id"
)
merged.to_csv(output, index=False)
print(f"\nSaved: {output}")


SUMMER — Per-class correlation (conf=0.25)

Total class correlation (all plots with >0 field plants):
  Total:
    Pearson r=0.665 (p=0.0000) *
    Spearman rho=0.660 (p=0.0000) *

Deciduous_tree correlation (plots with >0 field deciduous):
  deciduous_tree:
    Pearson r=0.649 (p=0.0001) *
    Spearman rho=0.723 (p=0.0000) *

Shrub correlation (plots with >0 field shrubs):
  shrub:
    Pearson r=0.770 (p=0.0000) *
    Spearman rho=0.789 (p=0.0000) *

Detection counts summary:
  Total field plants:   565
  Total model preds:    1366
  Field deciduous:      323
  Pred deciduous:       682
  Field shrub:          242
  Pred shrub:           684

FALL — Per-class correlation (conf=0.25)

Total class correlation (all plots with >0 field plants):
  Total:
    Pearson r=0.416 (p=0.0036) *
    Spearman rho=0.550 (p=0.0001) *

Deciduous_tree correlation (plots with >0 field deciduous):
  deciduous_tree:
    Pearson r=0.522 (p=0.0037) *
    Spearman rho=0.702 (p=0.0000) *

Shrub correlation (p